# t-stack-trajectory — ex2: window-slice a (B,T,D) trajectory and return per-step deltas

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `t-stack-trajectory`. Running the final beacon cell reports progress against the `Generative: torch.stack trajectory` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Generative: torch.stack trajectory` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`t-stack-trajectory`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "t-stack-trajectory"
DD_SUBTOPIC = "Generative: torch.stack trajectory"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `torch.stack` trajectory — deepening refresher

A trajectory `(B, T, D)` is the canonical layout from stacking a Python list of `(B, D)` per-step latents with `t.stack(..., dim=1)`. Once you have the trajectory, three operations dominate downstream analysis:

1. **Window slicing.** `traj[:, t0:t1, :]` extracts a contiguous time window. Shape becomes `(B, t1 - t0, D)`.
2. **Per-step deltas.** `delta = traj[:, 1:, :] - traj[:, :-1, :]` gives the per-step changes. Shape collapses from `T` to `T - 1` along the time axis — every other axis is preserved.
3. **Reductions over time.** `traj.mean(dim=1)` averages along the time axis to give a `(B, D)` summary. `traj.std(dim=1)` gives per-dim volatility.

**Why `dim=1` everywhere.** Once you've committed to `(B, T, D)` (the standard sequence convention), time slicing, diffing, and reducing are all `dim=1` operations. Mixing `dim=0` operations on a `(B, T, D)` tensor accidentally walks the batch axis — usually visible only as a weird-looking learning curve.

### Exercise 2 — window-slice a (B,T,D) trajectory and return per-step deltas

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply trajectory slicing (`traj[:, t0:t1, :]`) plus consecutive-difference (`win[:, 1:] - win[:, :-1]`) to extract a windowed per-step delta tensor of shape `(B, t1 - t0 - 1, D)`.
> Keywords: trajectory, slicing, delta, time-axis
> ```

**KCs targeted:** `slice-along-time-axis-dim1`, `consecutive-step-delta`

Implement `ex2_window_deltas(traj, t0, t1)`. Given a `(B, T, D)` trajectory tensor, extract the time window `[t0, t1)` (Python half-open convention) and return the per-step deltas WITHIN that window:

1. Slice: `window = traj[:, t0:t1, :]` — shape `(B, t1 - t0, D)`.
2. Compute deltas along the time axis: `deltas = window[:, 1:, :] - window[:, :-1, :]` — shape `(B, t1 - t0 - 1, D)`.
3. Return `deltas`.

**Edge cases.**
- `t1 - t0 == 1`: the window has a single timestep ⇒ deltas has shape `(B, 0, D)`. That's the natural empty case — return it as-is, do NOT raise.
- `t1 - t0 == 0`: empty window ⇒ deltas shape `(B, 0, D)`. (Slicing handles this — the subtraction is over empty slices.)
- `t0 == 0, t1 == T`: full-trajectory deltas, shape `(B, T - 1, D)`. This is the common case.

**Do NOT use a Python loop.** Slice + subtract is one tensor operation each.

Inputs:
- `traj`: `(B, T, D)` float tensor.
- `t0`, `t1`: ints with `0 <= t0 <= t1 <= T`.

Output: `(B, t1 - t0 - 1, D)` float tensor (with `max(0, ...)`).

In [ ]:
def ex2_window_deltas(traj: Tensor, t0: int, t1: int) -> Tensor:
    """Slice [t0:t1) along time, then return per-step deltas."""
    raise NotImplementedError()


def _test_ex2():
    # Tiny exact case.
    traj = t.tensor([
        [[1.0, 2.0], [3.0, 4.0], [6.0, 8.0], [10.0, 14.0]],
        [[0.0, 0.0], [1.0, 1.0], [3.0, 3.0], [6.0,  6.0]],
    ])  # (B=2, T=4, D=2)

    # Full window — all consecutive deltas.
    d_full = ex2_window_deltas(traj, 0, 4)
    assert d_full.shape == (2, 3, 2), f'expected (2,3,2), got {tuple(d_full.shape)}'
    expected_full = t.tensor([
        [[2.0, 2.0], [3.0, 4.0], [4.0, 6.0]],
        [[1.0, 1.0], [2.0, 2.0], [3.0, 3.0]],
    ])
    assert t.allclose(d_full, expected_full), f'full-window deltas wrong:\n{d_full}'

    # Sub-window [1, 3) → 2 timesteps → 1 delta.
    d_sub = ex2_window_deltas(traj, 1, 3)
    assert d_sub.shape == (2, 1, 2), f'expected (2,1,2), got {tuple(d_sub.shape)}'
    expected_sub = t.tensor([
        [[3.0, 4.0]],   # 6-3, 8-4
        [[2.0, 2.0]],   # 3-1, 3-1
    ])
    assert t.allclose(d_sub, expected_sub), f'sub-window deltas wrong:\n{d_sub}'

    # Single-timestep window → empty deltas.
    d_one = ex2_window_deltas(traj, 2, 3)
    assert d_one.shape == (2, 0, 2), f'single-step window must give 0 deltas, got {tuple(d_one.shape)}'

    # Empty window → also empty deltas.
    d_empty = ex2_window_deltas(traj, 2, 2)
    assert d_empty.shape == (2, 0, 2), f'empty window must give shape (B,0,D), got {tuple(d_empty.shape)}'

    # Realistic shape.
    rng = t.Generator().manual_seed(0)
    B, T, D = 8, 30, 6
    big = t.randn(B, T, D, generator=rng)
    d = ex2_window_deltas(big, 5, 20)
    assert d.shape == (B, 14, D), f'expected ({B},14,{D}), got {tuple(d.shape)}'
    # Spot check.
    expected_first = big[:, 6, :] - big[:, 5, :]
    assert t.allclose(d[:, 0, :], expected_first, atol=1e-6), 'first delta mismatch'
    expected_last = big[:, 19, :] - big[:, 18, :]
    assert t.allclose(d[:, -1, :], expected_last, atol=1e-6), 'last delta mismatch'

    # Type preservation.
    assert d.dtype == big.dtype, f'dtype must propagate, got {d.dtype} vs {big.dtype}'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_window_deltas(traj, t0, t1):
    window = traj[:, t0:t1, :]
    return window[:, 1:, :] - window[:, :-1, :]
```

**Why slicing handles all the edge cases for free.** `traj[:, 2:2, :]` gives a `(B, 0, D)` tensor — empty along `dim=1`. `window[:, 1:, :]` and `window[:, :-1, :]` on an empty window are also empty; their difference is `(B, 0, D)`. Single-step `[2:3]` gives `(B, 1, D)`; `[1:]` is `(B, 0, D)` and `[:-1]` is `(B, 0, D)`; difference is `(B, 0, D)`. No branches needed.

**Why `dim=1` slicing, not `dim=0`.** The `(B, T, D)` convention puts time at `dim=1`. Slicing `traj[t0:t1]` (without the `:`) would slice the BATCH axis — silently wrong and a common bug. Always write the full `traj[:, t0:t1, :]` or `traj[:, t0:t1]` for clarity.

**Generalization.** The same `x[..., 1:] - x[..., :-1]` pattern computes consecutive deltas along ANY trailing axis. For `(B, T, D)`, you want `dim=1`; for a sequence at `dim=0` use `x[1:] - x[:-1]`. PyTorch also offers `t.diff(x, dim=1)` which is a one-line wrapper for the same op.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()